# 02 · What the network predicts, and the loss

The reverse step is (approximately) Gaussian, so we only need its mean. DDPM's
insight: rather than predict the mean directly, **predict the noise**
$\varepsilon$ that was added.

Since we *know* the true $\varepsilon$ (we added it ourselves), training is plain
regression:

$$\mathcal L=\mathbb E_{x_0,t,\varepsilon}\big\|\varepsilon-\varepsilon_\theta(x_t,t)\big\|^2$$

You'll build three things: the **time embedding**, the **denoiser**, and the
**loss**.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each one is followed by a
> **self-check** cell — run it and it will tell you if your implementation is
> correct (it compares against the reference and asserts the key properties).
>
> **Stuck?** The answer key is `solutions/notebooks/` (with plots), and the
> reference implementation lives in the `nanodiffusion/` package. Peeking is
> allowed — but try first.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from nanodiffusion.utils import pick_device, set_seed
from nanodiffusion.data import toy2d
from nanodiffusion.schedules import NoiseSchedule
from nanodiffusion.forward import add_noise          # you built this in nb 01
# references, used ONLY by the self-check cells:
from nanodiffusion.models import SinusoidalTimeEmbedding as ReferenceTimeEmbedding

set_seed(0)
device = pick_device()
print("device:", device)
data = toy2d("swiss_roll", 8000).to(device)
schedule = NoiseSchedule.make("cosine", 200).to(device)

## TODO 1 — sinusoidal time embedding

The denoiser must behave differently at different noise levels, so the timestep
$t$ is a real input. We encode it like transformer positions — sinusoids at many
frequencies — giving a smooth encoding of "how noisy is this?".

For embedding dim $d$ (even), with `half = d // 2`:

- frequencies: `freqs[i] = exp(-log(10000) * i / (half - 1))` for `i` in `0..half-1`
- `args = t[:, None] * freqs[None, :]`   → shape `(B, half)`
- output: `concat([sin(args), cos(args)], dim=-1)` → shape `(B, d)`

In [ ]:
class MyTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0, "dim must be even"
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        '''(B,) timesteps -> (B, dim) embedding.'''
        half = self.dim // 2
        # TODO: build `freqs` (use torch.arange(half, device=t.device)),
        #       then args = t.float()[:, None] * freqs[None, :],
        #       then return torch.cat([sin, cos], dim=-1)
        raise NotImplementedError

In [ ]:
# ---- self-check 1 ----
emb = MyTimeEmbedding(64).to(device)
t_probe = torch.arange(0, 200, device=device)
out = emb(t_probe)
ref = ReferenceTimeEmbedding(64).to(device)(t_probe)
assert out.shape == (200, 64), f"shape {tuple(out.shape)} != (200, 64)"
assert torch.allclose(out, ref, atol=1e-5), "doesn't match the reference"
plt.figure(figsize=(7, 2.5)); plt.imshow(out.detach().cpu().T, aspect="auto", cmap="RdBu")
plt.xlabel("timestep t"); plt.ylabel("embedding dim"); plt.title("Your time embedding"); plt.show()
print("✅ TODO 1 correct")

## TODO 2 — the denoiser

A tiny MLP is plenty for 2D. `__init__` is given; implement `forward`:

1. embed the timestep with `self.time_mlp(t)` → `(B, time_embed_dim)`
2. concatenate it onto `x` along the last dim → `(B, 2 + time_embed_dim)`
3. run that through `self.net` → `(B, 2)` predicted noise

In [ ]:
class MyDenoiser(nn.Module):
    def __init__(self, data_dim=2, hidden=128, depth=4, time_embed_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            MyTimeEmbedding(time_embed_dim),
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.SiLU(),
        )
        layers = [nn.Linear(data_dim + time_embed_dim, hidden), nn.SiLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, data_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        '''Predict the noise eps in `x` at timesteps `t`. x:(B,2) t:(B,) -> (B,2)'''
        # TODO: implement (3 lines: embed t, concat, run net)
        raise NotImplementedError

In [ ]:
# ---- self-check 2 ----
model = MyDenoiser().to(device)
xb, tb = data[:16], torch.randint(0, 200, (16,), device=device)
out = model(xb, tb)
assert out.shape == xb.shape, f"output {tuple(out.shape)} should match input {tuple(xb.shape)}"
assert out.requires_grad, "output should be part of the autograd graph"
print(f"✅ TODO 2 correct — {sum(p.numel() for p in model.parameters()):,} parameters")

## TODO 3 — the loss

The heart of DDPM training, in four steps:

1. sample a random timestep per item: `torch.randint(0, T, (B,), device=...)`
2. noise the batch: `x_t, noise = add_noise(x0, t, schedule)`
3. predict: `pred = model(x_t, t)`
4. return `F.mse_loss(pred, noise)`

In [ ]:
def my_ddpm_loss(model, x0: torch.Tensor, schedule: NoiseSchedule) -> torch.Tensor:
    '''DDPM epsilon-prediction MSE loss for one batch of clean data.'''
    # TODO: implement (4 lines, per the steps above)
    raise NotImplementedError

In [ ]:
# ---- self-check 3: can it overfit one batch? ----
# The classic Karpathy debugging move: if the model can't memorize 128 points,
# something is broken.
set_seed(0)
probe = MyDenoiser().to(device)
opt = torch.optim.Adam(probe.parameters(), lr=2e-3)
batch = data[:128]

vals = []
for step in range(400):
    l = my_ddpm_loss(probe, batch, schedule)
    opt.zero_grad(); l.backward(); opt.step()
    vals.append(l.item())

# average over a window: the per-step loss is noisy because t is random each step
start, end = sum(vals[:20]) / 20, sum(vals[-20:]) / 20
print(f"loss: {start:.3f} -> {end:.3f}")
assert end < start, "loss did not decrease — check your loss function"
print("✅ TODO 3 correct — overfit check passed")

## Train for real

Now train the model on the full dataset with **your** loss.

In [ ]:
set_seed(0)
model = MyDenoiser().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

losses = []
for step in range(2000):
    idx = torch.randint(0, data.shape[0], (512,), device=device)
    loss = my_ddpm_loss(model, data[idx], schedule)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 400 == 0:
        print(f"step {step:4d}   loss {loss.item():.4f}")
print(f"final loss {losses[-1]:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(losses, alpha=0.35)
plt.plot(torch.tensor(losses).unfold(0, 50, 1).mean(1), color="C1", label="smoothed")
plt.xlabel("step"); plt.ylabel("MSE"); plt.legend(); plt.title("Training loss"); plt.show()

tail = sum(losses[-100:]) / 100      # smoothed, for the same reason as above
print(f"mean loss over last 100 steps: {tail:.4f}")
assert tail < 0.6, "expected the loss to settle well below 0.6"
print("✅ trained")

In [ ]:
import os
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/my_toy_mlp.pt")
print("saved ../checkpoints/my_toy_mlp.pt — your very own trained diffusion model")

✅ **Done.** You have a trained ε-predictor. Compare with
`nanodiffusion/models/` and `nanodiffusion/objectives.py`.

Next: notebook 03 — use it to walk from noise back to data.